In [1]:
# --- PASO 0: LIMPIEZA TOTAL Y REPARACIÓN ---
import os
import time
import re  
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# 1. FORZAR LA PANTALLA PARA noVNC
os.environ["DISPLAY"] = ":99"  

# Cierra procesos viejos
os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.*")
os.system("rm -rf /tmp/.org.chromium.Chromium.*")
print("🧹 Limpieza completada. Pantalla virtual configurada.")

# --- VARIABLES GENERALES ---
NOMBRE_GRUPO = "G2"
propiedades_basicas = [] # Lista temporal
datos_finales = []       # Lista final
driver = None        

# --- PASO 1: CONFIGURACIÓN DEL NAVEGADOR ---
options = Options()
# options.add_argument("--headless=new") 

options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--disable-software-rasterizer")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

try:
    driver = webdriver.Chrome(options=options)
    print("🚀 Navegador iniciado. Iniciando Fase 1: Barrido de Enlaces...")

    # --- FASE 1: BARRIDO DE PÁGINAS PRINCIPALES ---
    limite_paginas = 3
    url_yapo = "https://www.yapo.cl/bienes-raices-alquiler-apartamentos/chile-es-coquimbo?_gl=1*xrdyl5*_gcl_au*MjQxMDY4NDMuMTc3NjU1NTI1MQ.."
    driver.get(url_yapo)

    for nivel_pagina in range(limite_paginas):
        print(f"\n--- 📄 Extrayendo tarjetas - Página {nivel_pagina + 1} ---")

        WebDriverWait(driver, 20).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".d3-ad-tile__content"))
        )
        
        # Scroll Humano
        for _ in range(5):
            driver.execute_script("window.scrollBy(0, 800);")
            time.sleep(1)
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(3) 

        bloques = driver.find_elements(By.CSS_SELECTOR, ".d3-ad-tile__content")

        for bloque in bloques:
            try:
                nombre = bloque.find_element(By.CSS_SELECTOR, ".d3-ad-tile__title").get_attribute("textContent")
                precio_texto = bloque.find_element(By.CSS_SELECTOR, ".d3-ad-tile__price").get_attribute("textContent")
                
                if not nombre or not precio_texto or not nombre.strip() or not precio_texto.strip():
                    continue

                try:
                    direccion = bloque.find_element(By.CSS_SELECTOR, ".d3-ad-tile__location").get_attribute("textContent")
                except:
                    direccion = "Dirección no especificada"

                # Extracción del enlace para la Fase 2
                enlace = bloque.find_element(By.CSS_SELECTOR, "a.d3-ad-tile__description").get_attribute("href")

                detalles = bloque.find_elements(By.CSS_SELECTOR, ".d3-ad-tile__details-item")
                dormitorios_txt = "0"
                banos_txt = "0"
                estacionamientos_txt = "0"
                
                for det in detalles:
                    html_interno = det.get_attribute("innerHTML")
                    texto = det.get_attribute("textContent").strip()
                    if "#bed" in html_interno: dormitorios_txt = texto
                    elif "#bath" in html_interno: banos_txt = texto
                    elif "#parking" in html_interno: estacionamientos_txt = texto

                # Guardamos en la lista temporal
                propiedades_basicas.append({
                    "identificador": nombre.strip(),
                    "valor": precio_texto.strip(),
                    "direccion": direccion.strip(),
                    "dormitorios": dormitorios_txt,
                    "estacionamientos": estacionamientos_txt,
                    "banos": banos_txt,
                    "enlace": enlace,
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "grupo": NOMBRE_GRUPO
                })
                
            except Exception as e:
                continue

        try:
            btn_sig = driver.find_element(By.XPATH, "//a[contains(@class, 'pagination') and contains(text(), 'Siguiente')] | //a[contains(@class, 'next')]")
            driver.execute_script("arguments[0].click();", btn_sig)
            time.sleep(5)
        except:
            print("⚠️ No se encontró botón siguiente.")
            break

    print(f"\n✅ Fase 1 completada. Se encontraron {len(propiedades_basicas)} enlaces para analizar.")

    # --- FASE 2: INMERSIÓN (DEEP SCRAPING) ---
    print("\n🤿 Iniciando Fase 2: Extracción de Descripciones Largas...")
    
    for i, prop in enumerate(propiedades_basicas):
        try:
            print(f"   Revisando {i+1}/{len(propiedades_basicas)}: {prop['identificador'][:30]}...")
            driver.get(prop["enlace"])
            
            # Esperamos a que cargue el contenedor de la descripción
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, ".d3-property-about__text"))
            )
            
            # Capturamos el texto completo
            desc_larga = driver.find_element(By.CSS_SELECTOR, ".d3-property-about__text").get_attribute("textContent").strip()
            
            # Agregamos la descripción a nuestro diccionario y lo pasamos a la lista final
            prop["descripcion"] = desc_larga
            datos_finales.append(prop)
            
            # Pausa de cortesía para no saturar el servidor de Yapo
            time.sleep(2)
            
        except Exception as e:
            print(f"   ⚠️ Error al capturar descripción profunda. Usando 'Sin descripción'.")
            prop["descripcion"] = "Sin descripción"
            datos_finales.append(prop)

    print(f"\n📊 Extracción profunda terminada.")

except Exception as e:
    print(f"🚨 Error en Selenium: {e}")

finally:
    if driver is not None:
        try:
            driver.quit()
        except:
            pass

# --- PASO 3: GUARDAR EN MONGODB ---
try:
    if datos_finales:
        client = MongoClient("mongodb://mongodb:27017/", serverSelectionTimeoutMS=5000)
        db = client["PruebaYapo"]
        
        # NUEVO: Colección G5 para los datos con descripciones profundas
        coleccion = db["G5_PruebaRealState_Deep"] 

        for d in datos_finales:
            precio_str = str(d["valor"])
            precio_solo_numeros = re.sub(r"\D", "", precio_str) 
            d["valor"] = float(precio_solo_numeros) if precio_solo_numeros else 0.0
            
            d["dormitorios"] = int(''.join(filter(str.isdigit, str(d["dormitorios"]))) or 0)
            d["estacionamientos"] = int(''.join(filter(str.isdigit, str(d["estacionamientos"]))) or 0)
            d["banos"] = int(''.join(filter(str.isdigit, str(d["banos"]))) or 0)

        coleccion.insert_many(datos_finales)
        print(f"💾 ¡{len(datos_finales)} datos cargados en la colección '{coleccion.name}' de la BD '{db.name}' correctamente!")
    else:
        print("⚠️ No hay datos válidos para guardar.")

except Exception as e:
    print(f"🚨 Error en MongoDB: {e}")

🧹 Limpieza completada. Pantalla virtual configurada.
🚀 Navegador iniciado. Iniciando Fase 1: Barrido de Enlaces...

--- 📄 Extrayendo tarjetas - Página 1 ---

--- 📄 Extrayendo tarjetas - Página 2 ---

--- 📄 Extrayendo tarjetas - Página 3 ---

✅ Fase 1 completada. Se encontraron 90 enlaces para analizar.

🤿 Iniciando Fase 2: Extracción de Descripciones Largas...
   Revisando 1/90: Arriendo año corrido, 2D 2B “G...
   Revisando 2/90: DEPARTAMENTO Avenida Las Higue...
   Revisando 3/90: Arriendo año corrido, amoblado...
   Revisando 4/90: Departamento ARRIENDO AÑO CARR...
   Revisando 5/90: Se Arrienda Año Corrido 2 Dorm...
   Revisando 6/90: Departamento OPORTUNIDAD!! AÑO...
   Revisando 7/90: Departamento DEPARTAMENTO EN E...
   Revisando 8/90: Departamento  SOL DE PEÑUELAS ...
   Revisando 9/90: Departamento PACIFICO 3100 HER...
   Revisando 10/90: Departamento DOÑA ANITA ARRIEN...
   Revisando 11/90: Rebajado!!!! Arriendo Anual De...
   Revisando 12/90: DEPARTAMENTO COSTA PEÑUELA...
  

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lower, col, when, count, avg, round, format_number, concat, lit

# --- 1. REINICIO Y CONEXIÓN A G5 (DEEP SCRAPER) ---
try: 
    spark.stop()
except: 
    pass

print("🚀 Iniciando conexión Spark - MongoDB (Base de datos: PruebaYapo.G5_PruebaRealState_Deep)...")

spark = SparkSession.builder \
    .appName("Analisis_Amenidades_G5") \
    .config("spark.mongodb.read.connection.uri", "mongodb://mongodb:27017/PruebaYapo.G5_PruebaRealState_Deep") \
    .getOrCreate()

df = spark.read.format("mongodb").load()

# --- 2. PREPARACIÓN DE DATOS (ZONAS Y OUTLIERS) ---
# Clasificamos ciudad y filtramos precios irreales
df_base = df.withColumn("ciudad", 
    when(col("direccion").contains("La Serena"), "La Serena")
    .when(col("direccion").contains("Coquimbo"), "Coquimbo")
    .otherwise("Otras zonas")
).filter((col("valor") >= 150000) & (col("valor") <= 1500000))

# --- 3. TEXT MINING: EXTRACCIÓN DE AMENIDADES ---
# Juntamos título y descripción en minúsculas para buscar en todo el texto a la vez
df_text = df_base.withColumn("texto_completo", lower(concat(col("identificador"), lit(" "), col("descripcion"))))

# Creamos las variables booleanas (1 = Sí tiene, 0 = No tiene)
df_amenities = df_text.withColumn("gimnasio", 
    when(col("texto_completo").contains("gimnasio") | col("texto_completo").contains("gym"), 1).otherwise(0)
).withColumn("quinchos", 
    when(col("texto_completo").contains("quincho") | col("texto_completo").contains("asador"), 1).otherwise(0)
).withColumn("terraza", 
    when(col("texto_completo").contains("terraza") | col("texto_completo").contains("balcon") | col("texto_completo").contains("balcón"), 1).otherwise(0)
).withColumn("lavanderia", 
    when(col("texto_completo").contains("lavanderia") | col("texto_completo").contains("lavandería") | col("texto_completo").contains("logia"), 1).otherwise(0)
).withColumn("mascotas", 
    when(col("texto_completo").contains("mascota") | col("texto_completo").contains("pet friendly"), 1).otherwise(0)
)

# --- 4. FUNCIÓN GENERADORA DE REPORTES ---
# Esto evita que repitas el mismo bloque de código 5 veces
def generar_reporte(dataframe, columna_amenidad, titulo_reporte):
    print(f"\n{'='*50}")
    print(f"📊 Impacto de: {titulo_reporte} (La Serena vs Coquimbo)")
    print(f"{'='*50}")
    
    # Filtramos solo las dos ciudades principales
    reporte = dataframe.filter(col("ciudad").isin("La Serena", "Coquimbo")) \
        .groupBy("ciudad", columna_amenidad).agg(
            count("*").alias("cantidad_propiedades"),
            avg("valor").alias("precio_prom")
        ).withColumn(
            "precio_promedio",
            concat(lit("$"), format_number(round(col("precio_prom"), 0), 0))
        ).orderBy("ciudad", columna_amenidad)
        
    reporte.select("ciudad", columna_amenidad, "cantidad_propiedades", "precio_promedio").show(truncate=False)

# --- 5. IMPRESIÓN DE RESULTADOS ---
generar_reporte(df_amenities, "gimnasio", "Gimnasio")
generar_reporte(df_amenities, "quinchos", "Quinchos / Asaderas")
generar_reporte(df_amenities, "terraza", "Terraza / Balcón")
generar_reporte(df_amenities, "lavanderia", "Lavandería Interna")
generar_reporte(df_amenities, "mascotas", "Permite Mascotas")

🚀 Iniciando conexión Spark - MongoDB (Base de datos: PruebaYapo.G5_PruebaRealState_Deep)...

📊 Impacto de: Gimnasio (La Serena vs Coquimbo)
+---------+--------+--------------------+---------------+
|ciudad   |gimnasio|cantidad_propiedades|precio_promedio|
+---------+--------+--------------------+---------------+
|Coquimbo |0       |22                  |$562,273       |
|Coquimbo |1       |4                   |$602,500       |
|La Serena|0       |30                  |$635,667       |
|La Serena|1       |8                   |$754,375       |
+---------+--------+--------------------+---------------+


📊 Impacto de: Quinchos / Asaderas (La Serena vs Coquimbo)
+---------+--------+--------------------+---------------+
|ciudad   |quinchos|cantidad_propiedades|precio_promedio|
+---------+--------+--------------------+---------------+
|Coquimbo |0       |11                  |$543,636       |
|Coquimbo |1       |15                  |$586,667       |
|La Serena|0       |21                  |$662,